<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Benford's law and other business checks

**[Benford's law and other business checks](https://learning.nextia-ai.com/courses/anomaly-detection/m03/benfords-law-and-other-business-checks/)** · Anomaly Detection: Find What Is Unusual · Module 3, lesson 7 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** run the checks that auditors use on amounts: a first-digit test with Benford's law, amounts just below an approval limit, duplicates, round numbers and totals that do not reconcile. You will see what each check shows, and the data where Benford's law does not apply.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account. |
| **You should know** | Percentages and a p-value; the refunds tables, from [Meet the Larkfield data](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/). |
| **You do not need** | An audit tool: every check is a few lines of pandas. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m03/benfords-law-and-other-business-checks/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M03-L07-benfords-law-and-other-business-checks/benfords-law-and-other-business-checks-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `refunds.csv`, `refunds_agents.csv`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "refunds.csv": "e35d78db18584188a2bae7742720796cc43d3801fcfc29c0c77b3e766ce28484",
    "refunds_agents.csv": "1a883275f23988a40d99955e17f23ba132d98b7fb67912b0e197478cfff21782",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Why 1 comes first more often

> **Predict.** An amount starts at 10 dollars and grows 12% at each step, until it passes 100. Which first digit do you see most often?

> **Check.** You should see the digit 1 7 times.

In [ ]:
values = [round(10 * 1.12 ** t, 1) for t in range(21)]       # an amount that grows 12% a step, from 10 to 96.5
first = [int(str(v)[0]) for v in values]
print(values)
print("first digits:", pd.Series(first).value_counts().sort_index().to_dict())

From 10 to 20, the amount must double, so it stays a long time on a first digit of 1. From 90 to 100, it needs only 11%, so it passes 9 quickly. In data that grows by multiplication and spans several orders of magnitude, about 30% of the first digits are 1 and less than 5% are 9. This is **Benford's law**.

In [ ]:
BENFORD = np.log10(1 + 1 / np.arange(1, 10))       # the share of first digit d is log10(1 + 1/d)
print(pd.Series(BENFORD, index=range(1, 10)).round(4).to_string())

Run the next cell. It defines `benford_check()`, which measures the **mean absolute deviation (MAD)** between the observed shares and Benford's shares, and `nigrini()`, which reads the MAD with the limits of the auditor Mark Nigrini. It prints nothing.

In [ ]:
from scipy import stats

def first_digits(amounts):
    """The first digit of each amount of 1 dollar or more."""
    return np.array([int(f"{v:.2f}".lstrip("0.")[0]) for v in amounts if v >= 1])

def benford_check(amounts):
    """Observed shares of the first digits, and the mean absolute deviation (MAD) from Benford's law."""
    shares = np.bincount(first_digits(amounts), minlength=10)[1:] / len(first_digits(amounts))
    span = np.log10(np.quantile(amounts, 0.99) / np.quantile(amounts, 0.01))   # orders of magnitude
    return {"n": len(first_digits(amounts)), "MAD": round(float(np.mean(np.abs(shares - BENFORD))), 4),
            "orders of magnitude": round(float(span), 2), "shares": shares.round(4)}

def nigrini(mad):
    """Nigrini's limits for the first digit."""
    return "close" if mad < 0.006 else "acceptable" if mad < 0.012 else "marginal" if mad < 0.015 else "nonconformity"

In [ ]:
payments = pd.read_csv("data/payments.csv.gz", usecols=["amount"])
refunds = pd.read_csv("data/refunds.csv")
agents = pd.read_csv("data/refunds_agents.csv")
print(f"{len(payments):,} payments, {len(refunds):,} refunds, {len(agents):,} agent-days")

## 2. Benford on Larkfield's amounts

> **Predict.** Payments, refunds, the refunds of agent A-29, and payments under 5 dollars. Which of them follow Benford's law?

> **Check.** You should see a MAD of 0.0215 for the payments and 0.0449 for A-29.

In [ ]:
checks = {"payments": payments.amount, "refunds": refunds.amount,
          "refunds of A-29": refunds.amount[refunds.agent_id == "A-29"],
          "payments under 5 dollars": payments.amount[payments.amount < 5]}
rows = []
for name, amounts in checks.items():
    b = benford_check(amounts.to_numpy())
    rows.append({"amounts": name, "n": b["n"], "MAD": b["MAD"], "Nigrini": nigrini(b["MAD"]),
                 "orders of magnitude": b["orders of magnitude"], "first digit 1": b["shares"][0], "first digit 9": b["shares"][8]})
benford_table = pd.DataFrame(rows)
four = "{:.4f}".format
print(benford_table.to_string(index=False, formatters={"MAD": four, "first digit 1": four, "first digit 9": four,
                                                       "orders of magnitude": "{:.2f}".format}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
digits = np.arange(1, 10)
ax.bar(digits - 0.2, benford_check(payments.amount.to_numpy())["shares"], width=0.4, label="payments")
ax.bar(digits + 0.2, benford_check(refunds.amount.to_numpy())["shares"], width=0.4, label="refunds")
ax.plot(digits, BENFORD, "k.-", label="Benford's law")
ax.set_xticks(digits)
ax.set_xlabel("first digit")
ax.set_ylabel("share")
ax.legend()
plt.show()

No group is "close". The payments are a **nonconformity** (MAD 0.0215), and nobody planted anything in the first digits of the payments. Payments are a mixture of basket sizes, mostly between 15 and 300 dollars, so their first digits do not follow Benford's curve. The payments under 5 dollars span less than one order of magnitude: Benford cannot apply to them at all. A-29's refunds have 18.1% of first digits 9: that is the one finding that points to a person.

Use Benford's law only on data that grows by multiplication, spans several orders of magnitude, and has no minimum, no maximum and no assigned numbers (such as prices or invoice numbers). Even then, a deviation is a reason to review, not a proof of fraud.

## 3. Just below the approval limit

A refund of 100 dollars or more needs a manager's approval. Refunds that cluster just below the limit can mean that someone avoids the approval. Run the next cell. It compares the share of refunds at 90 to 99.99 dollars of each agent with all agents, with a binomial test and Bonferroni for 40 agents.

In [ ]:
refunds["just_below"] = (refunds.amount >= 90) & (refunds.amount < 100)    # the approval limit is 100 dollars
share_all = refunds.just_below.mean()
per_agent = refunds.groupby("agent_id").just_below.agg(["mean", "sum", "size"])
per_agent["p"] = [stats.binomtest(int(k), int(n), share_all, alternative="greater").pvalue
                  for k, n in zip(per_agent["sum"], per_agent["size"])]
per_agent["after Bonferroni"] = per_agent.p < 0.05 / len(per_agent)
print(f"all agents: {share_all:.1%} of refunds at 90 to 99.99 dollars")
print(per_agent.sort_values("mean", ascending=False).head(5).round(4).to_string())

> **Check.** You should see A-29 first, at 16.1%, with `True` after Bonferroni.

A-29 has 16.1% of refunds just below the limit, against 3.5% for all agents (an anomaly we planted). This check asks the business question directly, so it is clearer than the first digits.

## 4. Duplicates and round numbers

In [ ]:
dups = refunds[refunds.duplicated(["order_id", "amount"], keep=False)].sort_values(["order_id", "date"])
print(f"{len(dups) // 2} pairs: the same order refunded twice with the same amount")
print(dups[["date", "agent_id", "order_id", "amount", "reason"]].head(6).to_string(index=False))
print(f"whole-dollar refunds: {(refunds.amount % 1 == 0).mean():.2%}")

Each pair is a refund paid twice for the same order (planted). Whole-dollar refunds are rare here: a team that types round amounts by hand would show more.

## 5. Totals that do not reconcile

The daily feed gives each agent's total refund amount. The refunds table has each refund. The two must agree.

In [ ]:
sums = refunds.groupby(["date", "agent_id"]).amount.sum().round(2)
feed = agents.set_index(["date", "agent_id"]).refund_amount        # the daily feed's total per agent
diff = (feed - sums.reindex(feed.index).fillna(0)).round(2)
bad = diff[diff.abs() > 0.01]
print(f"{len(bad)} agent-days where the feed total does not match the refunds")
print(bad.groupby(level="date").agg(["size", "sum"]).to_string())

> **Check.** You should see 21 agent-days.

On 2025-11-10, the feed sent zeros: every agent who refunded that day shows a negative difference. The other three are single agent-days where the feed and the refunds disagree (planted). A reconciliation finds broken data, not only fraud.

## 6. Check yourself

**Your turn 1.** What share of first digits does Benford's law give to the digit 2? Round to 4 decimals and write it as `benford_2`.

In [ ]:
benford_2 = ...   # your code

In [ ]:
expect('benford_2', benford_2, 0.1761, tolerance=0.0001)

**Your turn 2.** Which agent has the second-highest share of refunds at 90 to 99.99 dollars? Write the agent ID as a string in `second_agent`. Is that agent's p-value below the Bonferroni limit?

In [ ]:
second_agent = ...   # your code

In [ ]:
expect_hash('second_agent', second_agent, 'f8b6cd69893e777b')

**Your turn 3.** How many of the agent-days that do not reconcile are **not** on 2025-11-10? Write the number as `other_days`.

In [ ]:
other_days = ...   # your code

In [ ]:
expect('other_days', other_days, 3)

## Recap

- Benford's law fits data that grows by multiplication across several orders of magnitude. Larkfield's payments do not fit it (MAD 0.0215), and nothing is wrong with them.
- A direct check is clearer: A-29 has 16.1% of refunds just below the 100-dollar limit, against 3.5% for all agents.
- Duplicates, round numbers and totals that do not reconcile find paid-twice refunds and broken feeds. Each check is a reason to review, not a proof.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Far from the neighbours: kNN distance](https://learning.nextia-ai.com/courses/anomaly-detection/m04/far-from-the-neighbours-knn-distance/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m03/benfords-law-and-other-business-checks/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.